In [ ]:
import os
import re
import pathlib
import unicodedata
import pandas as pd
from rapidfuzz import fuzz

In [ ]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

# Nettoyage

In [ ]:
dataframe_product_detail = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export.xlsx",
    )
)
dataframe_product_detail

In [ ]:
dataframe_product_detail.dtypes

In [ ]:
dataframe_product_detail = dataframe_product_detail.rename(
    columns={
        "DISTRIBUTEUR": "distributor",
        "SOURCE DONNEES": "data_source",
        "CODE PRODUIT": "product_code",
        "DESCRIPTION": "description",
        "MARQUE": "brand",
        "INDUSTRIEL": "industrial",
        "UF": "unit",
        "Qté Facture": "quantity",
        "Montant HT": "amount_ht"
    }
)
dataframe_product_detail

In [ ]:
dataframe_product_detail_distributor = dataframe_product_detail["distributor"].isna().sum()
dataframe_product_detail_distributor

In [ ]:
dataframe_product_detail.dropna(how="all", inplace=True)
dataframe_product_detail

In [ ]:
dataframe_product_detail.drop(index=719, inplace=True) # À modifier si jamais la ligne 719 n'est plus vide et prendre la dernière ligne
dataframe_product_detail

In [ ]:
# Add new column after data_source column
dataframe_product_detail.insert(
    dataframe_product_detail.columns.get_loc("data_source") + 1,
    "product_name",
    ""
)
dataframe_product_detail

In [ ]:
dataframe_product_detail["amount_ht"] = dataframe_product_detail["amount_ht"].round(2)
dataframe_product_detail

In [ ]:
dataframe_product_detail["unit"] = dataframe_product_detail["unit"].str.upper()

dictionary_unit = {
    # BOCAL
    "BCL": "BOCAL",
    # BIDON
    "BID": "BIDON",
    # BOUTEILLE
    "BLLE": "BOUTEILLE",
    # BOÎTE
    "BT": "BOÎTE",
    "BT.": "BOÎTE",
    "BTE": "BOÎTE",
    "BOITE": "BOÎTE",
    # BRIQUE
    "BRQ": "BRIQUE",
    # COFFRET
    "CO": "COFFRET",
    "COF": "COFFRET",
    # COLIS
    "COL": "COLIS",
    # FLACON
    "FLC": "FLACON",
    # PIÈCE
    "PI": "PIÈCE",
    # POCHE
    "PCH": "POCHE",
    # SEAU
    "SEA": "SEAU",
    # UNITÉ
    "U": "UNITÉ"
}

dataframe_product_detail["unit"] = dataframe_product_detail["unit"].replace(dictionary_unit)

In [ ]:
dataframe_product_detail["product_code"] = dataframe_product_detail["product_code"].str.upper()

dictionary_brand = {
    # HELLMANN'S
    "HELLEMANSQUEEZE": "HELLMANN'S SQUEEZE",
    "HELLMANNSQUEEZE": "HELLMANN'S SQUEEZE",
    # AMORA
    "SAVORA": "AMORASAVORA"
}

dataframe_product_detail["product_code"] = dataframe_product_detail["product_code"].replace(dictionary_brand)

description_contains_brand = ["AMORA", "HELLMANN'S", "KNORR", "MAILLE", "MAIZENA", "TABASCO", "LIPTON", "ELEPHANT"]

for brand in description_contains_brand:
    dataframe_product_detail.loc[
        dataframe_product_detail["description"].str.contains(brand, case=False, na=False) |
        dataframe_product_detail["product_code"].str.contains(brand, case=False, na=False),
        "brand"
    ] = brand

In [ ]:
for column in dataframe_product_detail.select_dtypes(include=["object"]):
    dataframe_product_detail[column] = dataframe_product_detail[column].str.title().str.replace(r"(?<=')([A-Z])", lambda value: value.group(0).lower(), regex=True)

In [ ]:
dataframe_product_detail

In [ ]:
dataframe_product_detail.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    ),
    index=False
)

# Score de similarité

In [ ]:
product_file = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    )
)
mapping_file = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "mapping",
        "mapping_product.xlsx",
    ),
    sheet_name="mapping_products"
)

In [ ]:
def normalize_string(param_text: str) -> str:
    """ Delete accents and convert to uppercase """
    
    text = unicodedata.normalize("NFKD", param_text)
    text = text.encode("ASCII", "ignore").decode("utf-8")
    text = text.replace("'", "")
    text = re.sub(r"\s+", " ", text).strip()
    text = text.upper()
    text = re.sub(r"(\d+)\s*(ML|L|KG|G)", r"\1\2", text)

    return text

In [ ]:
def get_volume_category(param_text: str) -> str:
    """ Get the volume category based on thresholds """
    
    text_upper = param_text.upper()

    # ML
    match_ml = re.search(r"(\d+)\s*ML", text_upper)
    if match_ml:
        volume = int(match_ml.group(1))
        if volume <= 10:
            return "DOSETTE"
        elif volume <= 80:
            return "MINI"
        elif volume <= 200:
            return "150ML"
        elif volume <= 500:
            return "350ML"
        else:
            return "GRANDFORMAT"

    # KG
    match_kg = re.search(r"(\d+[,.]?\d*)\s*KG", text_upper)
    if match_kg:
        volume = float(match_kg.group(1).replace(",", "."))
        if volume > 1:
            return "1KG"

    # G
    match_g = re.search(r"(\d+)\s*G(?![A-Z])", text_upper)
    if match_g:
        volume = int(match_g.group(1))
        if volume <= 500:
            return "340G"
        else:
            return "700G"

    return ""

In [ ]:
VOLUME_CATEGORY_BRANDS = {"TABASCO", "MAIZENA"}

def find_product_name(param_row):
    """ Find the product name based on the description and brand of the product """

    raw_description = str(param_row["description"])
    description_product = normalize_string(raw_description)
    brand_product = normalize_string(str(param_row["brand"]))

    if any(brand in brand_product for brand in VOLUME_CATEGORY_BRANDS):
        volume_category = get_volume_category(raw_description)
        if volume_category:
            description_product += f" {volume_category}"

    best_score = 0
    best_product_name = None

    for _, rule in mapping_file.iterrows():
        # Split and normalize the brand keywords
        keywords_brands = [
            normalize_string(keyword_brand)
            for keyword_brand in str(rule["keywords_brands"]).split(";")
            if keyword_brand.strip()
        ]

        # If the brand keywords do not match, skip to the next rule
        if not any(keyword_brand in brand_product for keyword_brand in keywords_brands):
            continue

        # Split and normalize the other keywords
        keywords_others = [
            normalize_string(keyword_other)
            for keyword_other in str(rule["keywords_others"]).split(";")
            if keyword_other.strip()
        ]

        # Count how many keywords are found in the description
        matched = 0
        for keyword_other in keywords_others:
            if keyword_other in description_product:
                matched += 1

        # Keep the rule with the highest number of matched keywords
        if matched > 0 and matched > best_score:
            best_score = matched
            best_product_name = rule["product_name"]

    return best_product_name

product_file["product_name"] = product_file.apply(find_product_name, axis=1)

In [ ]:
product_file.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_final.xlsx",
    ),
    index=False
)